In [ ]:
## Learn Specific data science libraries like Pandas for data cleaning and manipulation. - Preprocessing Techniques - 1

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.feature_extraction.text import CountVectorizer
from IPython.display import display, HTML

In [39]:
# ==========================================
# 0. SETUP: Create the Expanded Dataset
# ==========================================
raw_data = {
    'Emp_ID':[101, 102, 103, 104, 105, 106, 107, 108, 109, 110],

    'Join_Date': [
        '2024-01-15', '2023-06-20', '2025-03-10', '2024-11-05', '2022-08-14',
        '2021-04-12', '2026-02-28', '2023-10-01', '2025-07-19', '2020-11-30'
    ],
    
    'Department': [
        'Tech', 'HR', 'Tech', 'Sales', None, 
        'Finance', 'tech', 'Sales', 'Finance', None
    ],  # 'tech' is lowercase to test text cleaning; another None added
    
    'Experience_Yrs':[2, 5, 1, 7, 10, 4, 1, 6, 3, 20],
    
    'Current_Salary':[65000, 80000, 72000, 95000, 115000, 88000, 60000, 92000, 85000, 250000], 
    # $250,000 acts as a massive executive salary outlier for scaling/outlier treatment
    
    'Stock_Options_Granted':[5000, 12000, 0, 25000, 40000, 8000, 0, 18000, 5000, 150000], 
    # Maintains the highly skewed nature of financial equity distributions
    
    'Performance_Score': [88, 92, None, 78, 95, 90, 82, None, 85, 98], 
    # Added more missing values (None) to practice imputation pipelines
    
    'Review_Notes': [
        'Great coder team player', 
        'Excellent recruiter', 
        'Needs training raw logic', 
        'Solid sales closer', 
        'Amazing director leader',
        'Accurate auditor meticulous records',
        'Fast learner python coding focus',
        'Strong negotiator relationship closer',
        'Budget planning strategic mindset',
        'Visionary executive scaling mastery'
    ]
}

df = pd.DataFrame(raw_data)
print("--- ORIGINAL DATASET ---")
print(df[['Emp_ID', 'Department', 'Experience_Yrs', 'Current_Salary', 'Performance_Score']])
print("\n" + "="*60 + "\n")

--- ORIGINAL DATASET ---
   Emp_ID Department  Experience_Yrs  Current_Salary  Performance_Score
0     101       Tech               2           65000               88.0
1     102         HR               5           80000               92.0
2     103       Tech               1           72000                NaN
3     104      Sales               7           95000               78.0
4     105        NaN              10          115000               95.0
5     106    Finance               4           88000               90.0
6     107       tech               1           60000               82.0
7     108      Sales               6           92000                NaN
8     109    Finance               3           85000               85.0
9     110        NaN              20          250000               98.0




In [40]:
# ==========================================
# 1. HANDLING MISSING VALUES
# ==========================================
# Impute numerical columns with the median; categorical with a string label
df['Performance_Score'] = df['Performance_Score'].fillna(df['Performance_Score'].median())
df['Department'] = df['Department'].fillna('Missing_Dept')

In [41]:
df['Performance_Score']

0    88.0
1    92.0
2    89.0
3    78.0
4    95.0
5    90.0
6    82.0
7    89.0
8    85.0
9    98.0
Name: Performance_Score, dtype: float64

In [55]:
df['Department']

0            Tech
1              HR
2            Tech
3           Sales
4    Missing_Dept
5         Finance
6            tech
7           Sales
8         Finance
9    Missing_Dept
Name: Department, dtype: str

In [43]:
# ==========================================
# 2. DATE/TIME FEATURE EXTRACTION
# ==========================================
# Convert strings to datetime objects, then pull meaningful periods out
df['Join_Date'] = pd.to_datetime(df['Join_Date'])
df['Join_Year'] = df['Join_Date'].dt.year
df['Join_Month'] = df['Join_Date'].dt.month

In [44]:
# ==========================================
# 3. CREATING DERIVED FEATURES
# ==========================================
# Divide total salary by years of experience to find the financial value per year
df['Salary_Per_Year_Exp'] = df['Current_Salary'] / df['Experience_Yrs']

In [45]:
# ==========================================
# 4. BINNING
# ==========================================
# Group experience years into specific categorical career tiers
bins = [0, 3, 7, 15]
labels = ['Junior', 'Mid-Level', 'Senior']
df['Experience_Tier'] = pd.cut(df['Experience_Yrs'], bins=bins, labels=labels)

In [46]:
# ==========================================
# 5. ENCODING CATEGORICAL FEATURES
# ==========================================
# Convert text flags into numerical columns using One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False)
encoded_dept = encoder.fit_transform(df[['Department']])
dept_cols = encoder.get_feature_names_out(['Department'])
df[dept_cols] = encoded_dept

In [47]:
# ==========================================
# 6. SCALING
# ==========================================
# Adjust Current_Salary to have a mean of 0 and a variance of 1
scaler = StandardScaler()
df['Scaled_Salary'] = scaler.fit_transform(df[['Current_Salary']])

In [48]:
# ==========================================
# 7. LOG TRANSFORMATION
# ==========================================
# Apply log1p (log(1+x)) to fix high skewness in stock grant numbers
df['Log_Stock_Options'] = np.log1p(df['Stock_Options_Granted'])

In [49]:
# ==========================================
# 8. OUTLIER TREATMENT
# ==========================================
# Cap extreme values at a specific limit (e.g., upper limit = 130,000)
upper_limit = 130000
df['Capped_Salary'] = np.where(df['Current_Salary'] > upper_limit, upper_limit, df['Current_Salary'])

In [50]:
# ==========================================
# 9. INTERACTION FEATURES
# ==========================================
# Multiply score and experience to capture their blended output signal
df['Score_Exp_Interaction'] = df['Performance_Score'] * df['Experience_Yrs']

In [51]:
# ==========================================
# 10. POLYNOMIAL FEATURES
# ==========================================
# Create non-linear degree squared features for specific metrics
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_feats = poly.fit_transform(df[['Experience_Yrs']])
df['Experience_Squared'] = poly_feats[:, 1]  # Extracting index 1 (X^2)

In [52]:
# ==========================================
# 11. AGGREGATION FEATURES
# ==========================================
# Add a contextual metric mapping the average salary of the department group back to each employee
dept_avg_map = df.groupby('Department')['Current_Salary'].transform('mean')
df['Dept_Avg_Salary'] = dept_avg_map

In [53]:
# ==========================================
# 12. TEXT FEATURE ENGINEERING
# ==========================================
# Transform strings into a structured bag-of-words token count array
vectorizer = CountVectorizer(stop_words='english')
text_counts = vectorizer.fit_transform(df['Review_Notes']).toarray()
text_cols = [f"word_{w}" for w in vectorizer.get_feature_names_out()]
df[text_cols] = text_counts

In [54]:
# ==========================================
# RENDER FINAL COMPREHENSIVE OUTPUT
# ==========================================
print("--- ENGINEERED & TRANSFORMED FEATURES ---")
print(df[['Emp_ID', 'Join_Year', 'Experience_Tier', 'Scaled_Salary', 'Log_Stock_Options', 'Experience_Squared', 'word_great']])
print("\n" + "="*60 + "\n")

--- ENGINEERED & TRANSFORMED FEATURES ---
   Emp_ID  Join_Year Experience_Tier  Scaled_Salary  Log_Stock_Options  \
0     101       2024          Junior      -0.675033           8.517393   
1     102       2023       Mid-Level      -0.387377           9.392745   
2     103       2025          Junior      -0.540794           0.000000   
3     104       2024       Mid-Level      -0.099721          10.126671   
4     105       2022          Senior       0.283821          10.596660   
5     106       2021       Mid-Level      -0.233960           8.987322   
6     107       2026          Junior      -0.770919           0.000000   
7     108       2023       Mid-Level      -0.157252           9.798183   
8     109       2025          Junior      -0.291492           8.517393   
9     110       2020             NaN       2.872728          11.918397   

   Experience_Squared  word_great  
0                 4.0           1  
1                25.0           0  
2                 1.0           0  